# パッチ .npy の可視化

`.npy` ファイルをピアノロール画像としてノートブック内に表示します。

1. 下のセルで **`NPY_PATH`**（見たい `.npy`）と **`MODE`** を指定
2. セルを実行すると画像が表示されます
3. `SAVE_PNG = True` なら PNG も保存されます

| MODE | 内容 |
|------|------|
| `auto` | tonal=RGB, drum=グレースケール |
| `rgb` | チャンネル 0,1,2 を R,G,B |
| `composite` | 全チャンネル合成（白一色） |
| `gray` | グレースケール |

In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()

# --- ここを書き換えて実行 ---
NPY_PATH = NOTEBOOK_DIR / "data/patches/test1/bar0000_tonal.npy"
MODE = "auto"  # auto | rgb | composite | gray
SAVE_PNG = True
OUTPUT_PATH = None  # None なら .npy と同じ場所に .png

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from patch_to_image import (
    _detect_layout,
    default_output_path,
    npy_to_image,
)

npy_path = Path(NPY_PATH)
if not npy_path.exists():
    raise FileNotFoundError(f"ファイルが見つかりません: {npy_path}")

data = np.load(npy_path)
kind, chw = _detect_layout(data)
image = npy_to_image(data, mode=MODE)

resolved_mode = MODE if MODE != "auto" else ("rgb" if kind == "tonal" else "gray")

print(f"入力: {npy_path}")
print(f"種類: {kind}, shape: {tuple(chw.shape)}")
print(f"表示モード: {resolved_mode}")

fig, ax = plt.subplots(figsize=(14, 4))
array = np.array(image)
if image.mode == "RGB":
    ax.imshow(array, aspect="auto")
else:
    ax.imshow(array, cmap="gray", vmin=0, vmax=255, aspect="auto")

ax.set_xlabel("time (tick)")
ax.set_ylabel("pitch (high → top)")
ax.set_title(npy_path.name)
plt.tight_layout()
plt.show()

if SAVE_PNG:
    out = Path(OUTPUT_PATH) if OUTPUT_PATH else default_output_path(npy_path, resolved_mode)
    out.parent.mkdir(parents=True, exist_ok=True)
    image.save(out)
    print(f"保存: {out}")

## （任意）同じ bar の tonal / drum を並べて表示

下のセルで `BAR_INDEX` を指定すると、ペアを左右に表示します。

In [ ]:
BAR_INDEX = 0  # bar0000
PATCH_DIR = NOTEBOOK_DIR / "data/patches/test1"

tonal_path = PATCH_DIR / f"bar{BAR_INDEX:04d}_tonal.npy"
drum_path = PATCH_DIR / f"bar{BAR_INDEX:04d}_drum.npy"

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

for ax, path, title in [
    (axes[0], tonal_path, "tonal (RGB)"),
    (axes[1], drum_path, "drum (gray)"),
]:
    if not path.exists():
        ax.set_title(f"{title}\n(not found)")
        ax.axis("off")
        continue
    img = npy_to_image(np.load(path), mode="auto")
    arr = np.array(img)
    if img.mode == "RGB":
        ax.imshow(arr, aspect="auto")
    else:
        ax.imshow(arr, cmap="gray", vmin=0, vmax=255, aspect="auto")
    ax.set_title(f"{title}\n{path.name}")
    ax.set_xlabel("time")
    ax.set_ylabel("pitch")

plt.tight_layout()
plt.show()